# Lecture 19 - Linear Regression 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
plt.style.use('fivethirtyeight')

%matplotlib inline

In [ ]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.offline as po
po.init_notebook_mode()

In [ ]:
from scipy import optimize
import functools
import math

## Lecture Outline

- **Review:** Correlation and association strength  
- **Nearest Neighbor (NN) Regression:** Predict using nearby data points  
    - **Prediction Line:** Visual trend from averages  
    - **Correlation vs. NN Regression:** Association vs. prediction  
- **Linear Regression:** Best-fit line for linear relationships  
- **Making Predictions:** Apply linear models to new data



### Review - Correlation 

#### Standard Units

$$
\text{StandardUnits}(x) = \frac{x - \text{Mean}(x)}{\text{Stdev}(x)} 
$$

In [ ]:
def standard_units(x):
    "Convert any array of numbers to standard units."
    return (x - np.average(x)) / np.std(x)

### Correlation

$$
\begin{align}
r 
& = \text{Mean}\left(\text{StandardUnits}(x) *  \text{StandardUnits}(y)\right)\\
& = \frac{1}{n} \sum_{i=1}^n \text{StandardUnits}(x_i) *  \text{StandardUnits}(y_i)\\
& = \frac{1}{n}\sum_{i=1}^n \left( \frac{x_i - \text{Mean}(x)}{\text{Stdev}(x)} \right) * \left( \frac{y_i - \text{Mean}(y)}{\text{Stdev}(y)} \right) \\
\end{align}
$$

In [ ]:
def correlation(df, x, y):
    """Computes the correlation between columns x and y"""
    x_su = standard_units(df[x])
    y_su = standard_units(df[y])
    return np.mean(x_su * y_su)

### Nearest Neighbor Regression
- A method for prediction:
    - **Group** each *x* with similar (nearby) *x* values  
    - **Average** the corresponding *y* values for each group  
- For each *x* value, the prediction is the average of the *y* values in its nearby group.  
- The graph of these predictions is called the **graph of averages**.  
- If the association between *x* and *y* is **linear**, then the points in the graph of averages tend to fall on a **line**.


## Prediction Lines 

Let's build an intuition about the relationship between the slope of the nearest neighbor line and the correlation coefficient.

We will again use the heights data. 

In [ ]:
families = pd.read_csv('data/family_heights.csv')
parent_avgs = (families["father"] + families["mother"])/2
heights = pd.DataFrame({'Parent Average': parent_avgs, 
                        'Child': families['child']})
heights.sort_values('Parent Average', inplace=True)
heights.head(10)

We will build a slightly more robust Nearest Neighbor predictor. 

In [ ]:
def nn_heights(parent_average, window=0.5):
    lower_bound = parent_average - window
    upper_bound = parent_average + window
    similar_child_heights = (
        heights[(heights['Parent Average'] >= lower_bound) & (heights['Parent Average'] < upper_bound)]["Child"]
    )
    if len(similar_child_heights) == 0: #handle the case when there is no data
        return np.nan # nan = not a number , a special floating point "number"
    else:
        return np.mean(similar_child_heights)

#### Make predictions at many different parent heights not just the heights in the dataset. 

In [ ]:
test_heights = pd.DataFrame({"Parent Average": np.arange(61,74,0.2)})
test_heights["NN Prediction"] = (
    test_heights.apply(lambda row: nn_heights(row['Parent Average']), axis=1))

In [ ]:
# Plot it, using plotly 
fig = px.scatter(heights, x="Parent Average", y="Child", height=600)
fig.add_scatter(x=test_heights["Parent Average"], 
                y=test_heights["NN Prediction"], name="NN Prediction")

#### It will be easier to start in standard units. 

In [ ]:
# Transform the heights data to standard units
su_heights = pd.DataFrame({'Parent Average': standard_units(heights['Parent Average']), 
                           'Child': standard_units(heights['Child'])})

# Transform the predictions to standard units 
su_test_heights = (
    pd.DataFrame({'Parent Average': 
                  (test_heights['Parent Average'] - heights['Parent Average'].mean()) 
                  / heights['Parent Average'].std(), 
                  'NN Prediction': 
                  (test_heights['NN Prediction'] - heights['Child'].mean()) 
                  / heights['Child'].std()}))

# Plot it 
fig = px.scatter(su_heights, x="Parent Average", y="Child", height=600)
fig.add_scatter(x=su_test_heights["Parent Average"], 
                y=su_test_heights["NN Prediction"], name="NN Prediction")    


#### Computing the correlation we get: 

In [ ]:
correlation(heights, "Parent Average", "Child")

What happens if we draw a line with that slope: 

In [ ]:
r = correlation(su_heights, "Parent Average", "Child")
fig = px.scatter(su_heights, x="Parent Average", y="Child", height=600)
fig.add_scatter(x=su_test_heights["Parent Average"], 
                y=su_test_heights["NN Prediction"], 
                name="NN Prediction")
fig.add_scatter(x=np.arange(-3,4,0.1), y= r * np.arange(-3,4,0.1), 
                name=f"Line(y={np.round(r,4)} x)")

### The Relationship between Correlations and NN Predictions

Here we examine the relationship between the nearest neighbor prediction "line" and the correlation for several synthetic datasets.


In [ ]:
def make_correlated_data(r, n=500):
    "Generate a a table with columns x and y with a correlation of approximately r"
    x = np.random.normal(0, 1, n)
    z = np.random.normal(0, 1, n)
    # This is "magic" to sample from a multivariate Gaussian
    y = r*x + (np.sqrt(1-r**2))*z 
    return pd.DataFrame({"x": x, "y": y})

In [ ]:
#  You don't need to understand all the parts of this function.
def make_correlation_and_line_plot(r):
    """ 
    Generates a plot of synthetic data with a correlation coefficient r
    along with the nearest neighbor predictions and 
    a line with the slope r and intercept 0
    """
    # Make synthetic data
    example = make_correlated_data(r).sort_values("x")
    
    # Compute nearest neighbor predictions
    def nn_prediction_example(x_val):
        """ Predicts y-value for x based on the example table """
        neighbors = (
            example[(example['x'] >= x_val - 0.25) & (example['x'] < x_val + 0.25)]["y"]
        )
        if len(neighbors) == 0:
            return np.nan
        else: 
            return np.mean(neighbors)   
    example["NN Prediction"] = (
        example.apply(lambda row: nn_prediction_example(row['x']), axis=1))
    
    # Generate Plots.
    x,y = example["x"], example["y"]
    fig = px.scatter(example, x="x", y="y", height=600)
    fig.add_scatter(x=example["x"], y=example["NN Prediction"], 
                    name="NN Prediction", line_color="red")
    fig.add_scatter(x=x, y= r * x, name=f"Line(y={r} x)")
    fig.add_scatter(x=x, y=x, line_color="gray", line_dash="dot", name="Line(y=x)")
    return fig

#### Correlation of 0.90

In [ ]:
make_correlation_and_line_plot(r=0.90)

#### Correlation of 0.60

In [ ]:
make_correlation_and_line_plot(...)

#### Correlation of 0.20

In [ ]:
make_correlation_and_line_plot(...)

#### Correlation of 0

In [ ]:
make_correlation_and_line_plot(...)

#### Correlation of -0.6

In [ ]:
make_correlation_and_line_plot(...)

# Defining the linear regression line

In standard units we developed a simple equation for the regression line:

\begin{align}
\text{SU}(y_\text{predicted}) = r * \text{SU}(x_\text{new})
\end{align}

where $r$ is the correlation coefficient and $\text{SU}$ is the standard units:

\begin{align}
\text{SU}(y_\text{predicted}) & = \frac{y_\text{predicted} - \text{Mean}(y)}{\text{Stdev}(y)} \\
\text{SU}(x_\text{new}) &= \frac{x_\text{new} - \text{Mean}(x)}{\text{Stdev}(x)}
\end{align}



Here we use $x_\text{new}$ to indicate a new $x$ value for which we want to make a prediction  $y_\text{predicted}$.

We would like to express this line in the original units of the data.  We can do that by substituting the definition of standard units:

\begin{align}
\frac{y_\text{predicted} - \text{Mean}(y)}{\text{Stdev}(y)} = r *  \frac{x_\text{new} - \text{Mean}(x)}{\text{Stdev}(x)}
\end{align}

While this equation does desribe a line it would look a little nicer in the form:

\begin{align}
y_\text{predicted} = \text{slope} * x_\text{new}  + \text{intercept}
\end{align}

Let's do some algebra to get that equation:
$$
\require{color}
\definecolor{comment}{RGB}{200,100,50}
\begin{align}
\frac{y_\text{predicted} - \text{Mean}(y)}{\text{Stdev}(y)} &= r *  \frac{x_\text{new} - \text{Mean}(x)}{\text{Stdev}(x)}\\
\frac{y_\text{predicted} - \text{Mean}(y)}{\text{Stdev}(y)} &= r * \frac{1}{\text{Stdev}(x)} x_\text{new} - r * \frac{1}{\text{Stdev}(x)}\text{Mean}(x)  & \color{comment} \text{Expanding the right side}\\
y_\text{predicted} - \text{Mean}(y) &= r * \frac{\text{Stdev}(y)}{\text{Stdev}(x)} x_\text{new} - r * \frac{\text{Stdev}(y)}{\text{Stdev}(x)}\text{Mean}(x) &  \color{comment} \text{Multiplying by $\text{Stdev}(y)$}\\
y_\text{predicted} &= r * \frac{\text{Stdev}(y)}{\text{Stdev}(x)} x_\text{new} + \text{Mean}(y) - r * \frac{\text{Stdev}(y)}{\text{Stdev}(x)}\text{Mean}(x) &  \color{comment} \text{Adding $\text{Mean}(y)$}\\
y_\text{predicted} &= \left(r * \frac{\text{Stdev}(y)}{\text{Stdev}(x)}\right) x_\text{new} + \left(\text{Mean}(y) - r * \frac{\text{Stdev}(y)}{\text{Stdev}(x)}\text{Mean}(x)\right) &  \color{comment} \text{Rearranging Terms}
\end{align}
$$

#### This means we can define the slope and intercept as:
\begin{align}
\text{slope} &= r * \frac{\text{Stdev}(y)}{\text{Stdev}(x)}\\
\text{intercept} & = \text{Mean}(y) - \text{slope} * \text{Mean}(x)
\end{align}

<br>

## Implementing Linear Regression

Using the above equations implement the slope and intercept functions. 

### Slope

In [ ]:
def slope(t, x, y):
    """Computes the slope of the regression line"""
    r = correlation(t, x, y)
    y_sd = np.std(t[y])
    x_sd = np.std(t[x])
    return r * y_sd / x_sd


In [ ]:
def intercept(t, x, y):
    """Computes the intercept of the regression line"""
    x_mean = np.mean(t[x])
    y_mean = np.mean(t[y])
    return y_mean - slope(t, x, y)*x_mean

#### Testing it out

In [ ]:
example = make_correlated_data(0.5)
slope(example, 'x', 'y')

#### Computing the slope and intercept for the heights dataset:

In [ ]:
heights_slope = slope(heights, 'Parent Average', 'Child')
heights_intercept = intercept(heights, 'Parent Average', 'Child')
[heights_slope, heights_intercept]

### Adding the regression predictions: 

In [ ]:
heights["Regression Prediction"] = heights_slope * heights["Parent Average"] + heights_intercept
heights.head(10)

In [ ]:
fig = px.scatter(heights, x="Parent Average", y="Child", height=600)
fig.add_scatter(x=test_heights["Parent Average"], 
                y=test_heights["NN Prediction"], name="NN Prediction")
line_name = f"y = {np.round(heights_slope,2)} x + {np.round(heights_intercept,2)}"
fig.add_scatter(x=heights["Parent Average"], 
                y=heights["Regression Prediction"],
                name=line_name)

### Predict Linear 

We can define a function to predict the y-value given a DataFrame and two columns, `x` and `y`. 

In [ ]:
def predict_linear(df, x, y):
    """Return an DataSeries of the regressions estimates at all the x values"""
    pred_y = slope(df, x, y) * df[x] + intercept(df, x, y)
    return pred_y

<br><br>

---


# Making Predictions with Linear Regression

- We can now compute predictions, but how good are they?
- How do we know that we have a good linear fit?
- To study this we will consider a new dataset.

In [ ]:
demographics = pd.read_csv('data/district_demographics2016.csv')
demographics.head(5)

In [ ]:
px.scatter(demographics, 
           x="College%", 
           y="Median Income",
           color="State")

In [ ]:
correlation(demographics, 'College%', 'Median Income')

**Discussion Question:** Any concerns about the correlation computation being done here?


<br><br>

## Making Predictions

#### Here we will try to predict the income for each district as a function of the percent of college educated people.

In [ ]:
demo_slope = slope(demographics, 'College%', 'Median Income')
demo_intercept = intercept(demographics, 'College%', 'Median Income')
print("Slope:", demo_slope)
print("Intercept:", demo_intercept)

#### Make the actual predictions.

In [ ]:
demographics["Linear Prediction"] =  predict_linear(demographics, 'College%', 'Median Income')
demographics.head(5)

#### Visualizing the predictions:

In [ ]:
fig = px.scatter(demographics, x="College%", y="Median Income", height=400)
fig.add_scatter(x=demographics["College%"], 
                y=demographics["Linear Prediction"], 
                name="Linear Prediction")

In [ ]:
fig = px.scatter(demographics, x="College%", y="Median Income", height=400)
xtest = np.arange(0, 75, 1)
fig.add_scatter(x=xtest, 
                y=demo_slope * xtest + demo_intercept,
                name = f"{np.round(demo_slope, 2)} x + {np.round(demo_intercept)}")
fig

# See You Next Lecture!